# 03 - Signals and trades

What the strategies see: per-bar multi-horizon consensus, confidence and variance spikes, next to
price and the trades they produced. Uses the same out-of-sample TEST block as 02.

In [1]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
STRATEGY = "calibrated_quantile"
WINDOW = 1500                   # bars to plot (the last WINDOW of the test block)

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from neural_trade.notebook import BacktestExplorer, pick_run

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
explorer = BacktestExplorer.from_run(run_dir, csv_path=CSV_PATH)
s = explorer.signals
res = explorer.run(STRATEGY, costs={"random_seeds": 0}, baselines=False)

run: ..\runs\20260924T114819Z-e23fd9f-dirty-af67ee43
CalibrationPipeline loaded from '..\runs\20260924T114819Z-e23fd9f-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


## Signal summary

In [3]:
features = pd.DataFrame({"weighted_direction": s.weighted_direction, "strength": s.strength,
                         "avg_confidence": s.avg_confidence, "agreement": s.agreement, "consensus": s.consensus,
                         "magnitude_coherent": s.magnitude_coherent, "direction_aligned": s.direction_aligned,
                         "var_spike": s.var_spike, "volatility_$": s.volatility})
features.describe().T

,count,mean,std,min,25%,50%,75%,max
weighted_direction,7236.0,0.501471,0.021478,0.398197,0.491430,0.503655,0.514939,0.609243
strength,7236.0,0.050367,0.030467,0.000988,0.028860,0.043334,0.064217,0.262887
avg_confidence,7236.0,0.509718,0.171887,0.051407,0.361300,0.528708,0.655899,0.804198
agreement,7236.0,0.354293,0.091938,0.333333,0.333333,0.333333,0.333333,1.000000
consensus,7236.0,-0.000276,0.516219,-1.000000,0.000000,0.000000,0.000000,1.000000
volatility_$,7236.0,229.680931,59.449309,130.692150,180.446816,221.040548,276.747410,476.063736


In [4]:
lo = max(0, len(s) - WINDOW)
x = np.arange(lo, len(s))
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, row_heights=[0.4, 0.2, 0.2, 0.2], vertical_spacing=0.03,
                    subplot_titles=("Close and trades", "Weighted P(up)", "Strength and confidence", "h1 variance"))
fig.add_trace(go.Scatter(x=x, y=s.close[lo:], name="close", line=dict(width=1)), 1, 1)
for t in [t for t in res.trades if t.entry_bar >= lo]:
    fig.add_trace(go.Scatter(x=[t.entry_bar, t.exit_bar], y=[t.entry_price, t.exit_price], mode="lines+markers",
                             line=dict(color="#15803d" if t.net_pnl > 0 else "#b91c1c", width=2),
                             showlegend=False, hovertext=f"{t.side} {t.exit_reason} {t.net_pnl:+.2f}"), 1, 1)
fig.add_trace(go.Scatter(x=x, y=s.weighted_direction[lo:], name="weighted P(up)"), 2, 1)
fig.add_hline(y=0.5, line_dash="dot", row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=s.strength[lo:], name="strength"), 3, 1)
fig.add_trace(go.Scatter(x=x, y=s.avg_confidence[lo:], name="confidence"), 3, 1)
fig.add_trace(go.Scatter(x=x, y=s.var_scaled[lo:, 1], name="var h1"), 4, 1)
fig.add_trace(go.Scatter(x=x[s.var_spike[lo:]], y=s.var_scaled[lo:, 1][s.var_spike[lo:]], mode="markers",
                         name="spike", marker=dict(color="#b45309", size=5)), 4, 1)
fig.update_layout(height=900, hovermode="x unified")
fig.show()

## Trades by exit reason

In [5]:
trades = res.trades_frame()
trades.groupby("exit_reason")[["net_pnl", "bars_held"]].agg(["count", "mean", "sum"]) if len(trades) else trades

net_pnl                         bars_held                
              count       mean          sum     count       mean  sum
exit_reason                                                          
REV              97 -16.785733 -1628.216096        97   6.463918  627
SL                2 -65.144937  -130.289874         2  12.000000   24
TIME             45 -22.399511 -1007.977979        45  15.000000  675